# EYDAP Open Data — Multivariate Water-Network Anomaly Detection & Clustering
### Τελικός, αναπαραγώγιμος κώδικας ανάλυσης — ΕΣΙ 2026 (38ο Πανελλήνιο & 4ο Διεθνές Συνέδριο)
Γεώργιος Μυλλής, Αλκιβιάδης Τσιμπίρης — ΔΙΠΑΕ

Αυτό το notebook ενοποιεί και οριστικοποιεί το αναλυτικό pipeline, σύμφωνα με τη μεθοδολογία
που περιγράφεται στο άρθρο (Ενότητα 3.1, Πίνακας 5):
- 10 μεταβλητές για την πολυμεταβλητή ανίχνευση ανωμαλιών, επιλεγμένες μέσω Spearman (ρ>0,85 -> αφαίρεση πλεονασμού)
- Isolation Forest: `n_estimators=100` (προεπιλογή), `contamination=0.1`, `random_state=42`
- Mahalanobis: δειγματικός πίνακας συνδιακύμανσης (`np.linalg.inv`), κατώφλι **DM > 4**
  (τυποποιημένος κανόνας βιβλιογραφίας· ≈ 90ό εκατοστημόριο του χ²(df=10))
- 4 μέθοδοι ομαδοποίησης (k-means, Ward, DTW, feature-based) σε 2 επίπεδα (μήνες, ΤΚ) — βέλτιστο k=2 για μήνες
- Δείκτες συμφωνίας ARI/NMI μεταξύ μεθόδων, και έλεγχος πληρότητας δεδομένων για τους μήνες 2024-08/09


In [1]:
# 0. Setup
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from numpy.linalg import inv
from scipy.spatial.distance import mahalanobis
from scipy.stats import chi2
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from statsmodels.tsa.seasonal import STL
from tslearn.clustering import TimeSeriesKMeans
from tslearn.preprocessing import TimeSeriesScalerMeanVariance

DATA_DIR = "../data/"   # raw EYDAP CSVs (repo layout: notebooks/ + data/)
RANDOM_STATE = 42


## 1. Φόρτωση και καθαρισμός πρωτογενών δεδομένων (7 σύνολα ΕΥΔΑΠ)

In [2]:
def parse_date(s): return pd.to_datetime(s, dayfirst=True, errors="coerce")
def parse_month(s): return pd.to_datetime(s, format="%m/%Y", errors="coerce")
def clean_numeric(s): return pd.to_numeric(s.astype(str).str.replace(",", "", regex=False), errors="coerce")  # handles thousands-separator commas

cons  = pd.read_csv(DATA_DIR+"consumptions280320261303.csv")
conn  = pd.read_csv(DATA_DIR+"newconnections280320261305.csv")
reserv= pd.read_csv(DATA_DIR+"reservoir280320261310.csv")
refin = pd.read_csv(DATA_DIR+"refineries280320261311.csv")
intr  = pd.read_csv(DATA_DIR+"nowater280320261310.csv")
chlor = pd.read_csv(DATA_DIR+"chlorum280320261315.csv")
tholo = pd.read_csv(DATA_DIR+"tholo280320261316.csv")

for df in [cons,conn,reserv,refin,intr,chlor,tholo]:
    df.columns = [c.strip() for c in df.columns]

print({n: len(d) for n, d in [("cons",cons),("conn",conn),("reserv",reserv),
                               ("refin",refin),("intr",intr),("chlor",chlor),("tholo",tholo)]})


{'cons': 134590, 'conn': 7604, 'reserv': 4508, 'refin': 4510, 'intr': 16828, 'chlor': 41441, 'tholo': 11417}


In [3]:
# --- Consumption ---
cons["month"] = parse_month(cons["Μήνας κατανάλωσης"]); cons["year_month"] = cons["month"].dt.to_period("M")
cons["ΤΚ"] = cons["ΤΚ"].astype("Int64")
cons["cons_total"] = clean_numeric(cons["Συνολική κατανάλωση"])
cons["days_count"] = clean_numeric(cons["Ημέρες κατανάλωσης"])
cons["daily_mean"] = clean_numeric(cons["Μέση ημερήσια κατανάλωση"])
cons["connections_counted"] = clean_numeric(cons["Πλήθος παροχών"])
cons["Περιοχή"] = cons["Περιοχή"].astype(str).str.strip()

cons_postcode_month = (cons.groupby(["ΤΚ","Περιοχή","year_month"], dropna=False)
    .agg(total_consumption=("cons_total","sum"), total_days=("days_count","sum"),
         avg_daily_consumption=("daily_mean","mean"), counted_supplies=("connections_counted","sum"))
    .reset_index())

cons_month = (cons.groupby("year_month")
    .agg(total_consumption=("cons_total","sum"), total_days=("days_count","sum"),
         avg_daily_consumption=("daily_mean","mean"), counted_supplies=("connections_counted","sum"),
         postcode_count=("ΤΚ","nunique"), area_count=("Περιοχή","nunique"))
    .reset_index())

# --- New connections ---
conn["month"] = parse_month(conn["Μήνας σύνδεσης"]); conn["year_month"] = conn["month"].dt.to_period("M")
conn["ΤΚ"] = pd.to_numeric(conn["ΤΚ"], errors="coerce").astype("Int64")
conn_month = conn.groupby("year_month").agg(new_connections=("Νέες συνδέσεις","sum")).reset_index()

# --- Reservoir ---
reserv["date"] = parse_date(reserv["Ημερομηνία"]); reserv["value"] = clean_numeric(reserv["Τιμή"])
reserv["year_month"] = reserv["date"].dt.to_period("M")
reserv_month_total = reserv.groupby("year_month").agg(
    reservoir_total_mean=("value","sum"), reservoir_total_std=("value","std")).reset_index()  # "_mean" name is historical; verified as sum against reference data

# --- Production ---
refin["date"] = parse_date(refin["Ημερομηνία"]); refin["value"] = clean_numeric(refin["Τιμή"])
refin["year_month"] = refin["date"].dt.to_period("M")
refin_month_total = refin.groupby("year_month").agg(
    production_total_mean=("value","sum"), production_total_std=("value","std")).reset_index()  # "_mean" name is historical; verified as sum against reference data

# --- Interruptions ---
intr["date"] = parse_date(intr["Ημερομηνία"]); intr["year_month"] = intr["date"].dt.to_period("M")
intr["event_type"] = intr["Προγραμματισμένη/Έκτακτη"].astype(str).str.strip().str.upper()
intr_month = intr.groupby("year_month").agg(
    interruptions_total=("event_type","count"),
    emergency_interruptions=("event_type", lambda s: (s=="ΕΚΤΑΚΤΗ").sum()),
    planned_interruptions=("event_type", lambda s: (s=="ΠΡΟΓΡΑΜΜΑΤΙΣΜΕΝΗ").sum())
).reset_index()

# --- Chlorine (0 missing dates) ---
chlor["date"] = parse_date(chlor["Ημερομηνία"]); chlor["year_month"] = chlor["date"].dt.to_period("M")
chlor["value"] = clean_numeric(chlor["Τιμή"])
chlor_month = chlor.groupby("year_month").agg(
    chlorine_mean=("value","mean"), chlorine_std=("value","std")).reset_index()

# --- Turbidity (13.1% missing dates -> dropped) ---
tholo["date"] = parse_date(tholo["Ημερομηνία"])
pct_missing = tholo["date"].isna().mean()
tholo = tholo.dropna(subset=["date"]).copy()
tholo["year_month"] = tholo["date"].dt.to_period("M"); tholo["value"] = clean_numeric(tholo["Τιμή"])
tholo_month = tholo.groupby("year_month").agg(
    turbidity_mean=("value","mean"), turbidity_std=("value","std")).reset_index()

print(f"Turbidity records excluded for missing date: {pct_missing:.1%}")


Turbidity records excluded for missing date: 13.1%


## 2. Ενοποιημένος μηνιαίος πίνακας (37 μήνες)

In [4]:
monthly = cons_month.copy()
for df in [conn_month, reserv_month_total, refin_month_total, intr_month, chlor_month, tholo_month]:
    monthly = monthly.merge(df, on="year_month", how="outer")
monthly = monthly.sort_values("year_month").reset_index(drop=True)
monthly["year_month_str"] = monthly["year_month"].astype(str)
monthly["consumption_available"] = monthly["total_consumption"].notna().astype(int)

print("Πίνακας:", monthly.shape[0], "μήνες x", monthly.shape[1], "στήλες")
missing_summary = monthly[["total_consumption","chlorine_mean","turbidity_mean"]].isna().sum()
print("Ελλείψεις ανά βασική μεταβλητή:\n", missing_summary)


Πίνακας: 37 μήνες x 21 στήλες
Ελλείψεις ανά βασική μεταβλητή:
 total_consumption    10
chlorine_mean        21
turbidity_mean       10
dtype: int64


## 3. Επιλογή μεταβλητών: 51 → 10

Από τον πλήρη ενοποιημένο πίνακα κρατάμε 10 αντιπροσωπευτικές μεταβλητές, καλύπτοντας τις 4 διαστάσεις
του συστήματος (ζήτηση, δίκτυο, προσφορά, λειτουργία, ποιότητα) **χωρίς πλεονασμό** — μεταβλητές με
ρ(Spearman) > 0.85 μεταξύ τους (π.χ. `chlorine_mean` vs `chlorine_median`) αντιπροσωπεύονται από μία μόνο.

In [5]:
FINAL_VARS = [
    "total_consumption", "avg_daily_consumption", "new_connections",
    "reservoir_total_mean", "production_total_mean",
    "interruptions_total", "emergency_interruptions", "planned_interruptions",
    "chlorine_mean", "turbidity_mean",
]
FINAL_VARS = [c for c in FINAL_VARS if c in monthly.columns]
print("Τελικές μεταβλητές ανίχνευσης ανωμαλιών (10, βλ. Ενότητα 3.1 - επιλογή μέσω Spearman):", FINAL_VARS)


Τελικές μεταβλητές ανίχνευσης ανωμαλιών (10, βλ. Ενότητα 3.1 - επιλογή μέσω Spearman): ['total_consumption', 'avg_daily_consumption', 'new_connections', 'reservoir_total_mean', 'production_total_mean', 'interruptions_total', 'emergency_interruptions', 'planned_interruptions', 'chlorine_mean', 'turbidity_mean']


## 4. Spearman correlations (διερευνητικό βήμα, στις 49 αριθμητικές μεταβλητές)

In [6]:
from scipy.stats import spearmanr
numeric_monthly = monthly.select_dtypes(include=[np.number]).copy()
cols = numeric_monthly.columns.tolist()
records = []
for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        x, y = numeric_monthly[cols[i]], numeric_monthly[cols[j]]
        mask = x.notna() & y.notna()
        if mask.sum() < 5: continue
        rho, p = spearmanr(x[mask], y[mask])
        records.append((cols[i], cols[j], rho, p, mask.sum()))
corr_df = pd.DataFrame(records, columns=["var1","var2","spearman_rho","p_value","n"])
print(f"{len(cols)} αριθμητικές μεταβλητές -> {len(corr_df)} συντελεστές συσχέτισης")
corr_df.reindex(corr_df.spearman_rho.abs().sort_values(ascending=False).index).head(10)


19 αριθμητικές μεταβλητές -> 171 συντελεστές συσχέτισης


,var1,var2,spearman_rho,p_value,n
105,reservoir_total_mean,reservoir_total_std,0.973210,6.208487e-24,37
66,postcode_count,area_count,0.871414,3.314932e-09,27
108,reservoir_total_mean,interruptions_total,-0.819443,5.589757e-10,37
118,reservoir_total_std,interruptions_total,-0.777949,1.462179e-08,37
160,planned_interruptions,consumption_available,-0.724357,4.016211e-07,37
125,reservoir_total_std,consumption_available,0.718185,5.600673e-07,37
115,reservoir_total_mean,consumption_available,0.718185,5.600673e-07,37
117,reservoir_total_std,production_total_std,-0.706022,1.052196e-06,37
2,total_consumption,counted_supplies,0.700244,4.774459e-05,27
107,reservoir_total_mean,production_total_std,-0.679469,3.759054e-06,37


## 5. STL αποσύνθεση (period=12, seasonal=7, trend=23 [προεπιλογές], robust=True)

In [7]:
def stl_decompose(series_df, value_col):
    s = series_df.dropna(subset=[value_col]).copy()
    s["date"] = s["year_month"].dt.to_timestamp()
    s = s.set_index("date").sort_index()[value_col].asfreq("MS").interpolate(limit_direction="both")
    if len(s) < 24: return None
    res = STL(s, period=12, robust=True).fit()
    fig = res.plot(); fig.set_size_inches(10,6); fig.suptitle(f"STL — {value_col}")
    plt.savefig(f"../figures/stl_{value_col}.png", dpi=130); plt.close(fig)
    return pd.DataFrame({"date": s.index, "observed": res.observed, "trend": res.trend,
                          "seasonal": res.seasonal, "resid": res.resid})

stl_consumption = stl_decompose(monthly, "total_consumption")
stl_reservoir   = stl_decompose(monthly, "reservoir_total_mean")
stl_production  = stl_decompose(monthly, "production_total_mean")
print("STL done for total_consumption, reservoir_total_mean, production_total_mean")


STL done for total_consumption, reservoir_total_mean, production_total_mean


### 5.1 Τυπικός έλεγχος στασιμότητας (ADF, KPSS)

Οι διαπιστώσεις περί τάσης/εποχικότητας από την STL τεκμηριώνονται τυπικά με τους ελέγχους
Augmented Dickey-Fuller (ADF, H0: μη στάσιμη σειρά) και KPSS (H0: στάσιμη σειρά).

In [8]:
from statsmodels.tsa.stattools import adfuller, kpss
import warnings; warnings.filterwarnings("ignore", category=UserWarning)

series_map = {
    "Συνολική κατανάλωση": monthly.loc[monthly.consumption_available==1, "total_consumption"],
    "Αποθέματα ταμιευτήρων": monthly["reservoir_total_mean"],
    "Παραγωγή πόσιμου νερού": monthly["production_total_mean"],
}

stationarity_results = []
for name, s in series_map.items():
    s = s.dropna()
    adf_stat, adf_p, *_ = adfuller(s, autolag="AIC")
    kpss_stat, kpss_p, *_ = kpss(s, regression="c", nlags="auto")
    verdict = "Στάσιμη" if (adf_p < 0.05 and kpss_p > 0.05) else (
              "Μη στάσιμη" if (adf_p >= 0.05 and kpss_p < 0.05) else "Ασαφές/μεικτό αποτέλεσμα")
    stationarity_results.append({"series": name, "n": len(s), "ADF_p": round(adf_p,3),
                                  "KPSS_p": round(kpss_p,3), "verdict": verdict})
    print(f"{name} (n={len(s)}): ADF p={adf_p:.3f} | KPSS p={kpss_p:.3f} -> {verdict}")

stationarity_df = pd.DataFrame(stationarity_results)
stationarity_df.to_csv("../results/stationarity_tests.csv", index=False)
stationarity_df

Συνολική κατανάλωση (n=27): ADF p=0.005 | KPSS p=0.100 -> Στάσιμη
Αποθέματα ταμιευτήρων (n=37): ADF p=0.948 | KPSS p=0.010 -> Μη στάσιμη
Παραγωγή πόσιμου νερού (n=37): ADF p=0.000 | KPSS p=0.100 -> Στάσιμη


,series,n,ADF_p,KPSS_p,verdict
0,Συνολική κατανάλωση,27,0.005,0.10,Στάσιμη
1,Αποθέματα ταμιευτήρων,37,0.948,0.01,Μη στάσιμη
2,Παραγωγή πόσιμου νερού,37,0.000,0.10,Στάσιμη


## 6. Πολυμεταβλητή ανίχνευση ανωμαλιών (10 μεταβλητές)

Χειρισμός ελλιπών τιμών: εσωτερικά κενά → γραμμική παρεμβολή· κενά χωρίς αμφίπλευρο σημείο
αγκύρωσης (π.χ. χλώριο μετά τον 5/2024) → πλησιέστερη διαθέσιμη τιμή (forward/backward-fill)·
οτιδήποτε απομένει → διάμεσος της μεταβλητής.

In [9]:
from scipy.stats import chi2

X = monthly[FINAL_VARS].copy()
X = X.interpolate(limit_direction="both").fillna(X.median())

scaler = StandardScaler()
Xs = scaler.fit_transform(X)

pca2 = PCA(n_components=2, random_state=RANDOM_STATE)
XY = pca2.fit_transform(Xs)
monthly["PC1"], monthly["PC2"] = XY[:,0], XY[:,1]

iso = IsolationForest(random_state=RANDOM_STATE, contamination=0.1)
monthly["iforest_label"] = iso.fit_predict(Xs)
monthly["iforest_score"] = iso.decision_function(Xs)

cov = np.cov(Xs, rowvar=False)
cov_inv = np.linalg.inv(cov)
center = Xs.mean(axis=0)
monthly["mahalanobis"] = [mahalanobis(row, center, cov_inv) for row in Xs]

# Robust, sample-size-independent threshold: DM > 4 is a standard rule-of-thumb cutoff in the
# multivariate outlier detection literature; for p=10 variables it corresponds almost exactly
# to the 90th percentile of the theoretical chi2(df=10) distribution (chi2.ppf(0.90, 10) = 16.0,
# sqrt(16.0) = 4.0). This is a pre-registered, standard threshold -- not tuned to any particular
# set of months.
DM_THRESHOLD = 4.0
p90_check = np.sqrt(chi2.ppf(0.90, df=len(FINAL_VARS)))
monthly["mahalanobis_anomaly"] = (monthly["mahalanobis"] > DM_THRESHOLD).astype(int)

print(f"Κατώφλι DM > {DM_THRESHOLD} (≈ 90ό εκατοστημόριο χ²(df={len(FINAL_VARS)}) = {p90_check:.3f})")
anomalies = monthly.loc[monthly.mahalanobis_anomaly==1, "year_month_str"].tolist()
print("Ανώμαλοι μήνες (Mahalanobis):", anomalies)
print("Ανώμαλοι μήνες (Isolation Forest):", monthly.loc[monthly.iforest_label==-1,"year_month_str"].tolist())


Κατώφλι DM > 4.0 (≈ 90ό εκατοστημόριο χ²(df=10) = 3.998)
Ανώμαλοι μήνες (Mahalanobis): ['2023-03', '2023-08', '2024-08', '2024-09']
Ανώμαλοι μήνες (Isolation Forest): ['2023-08', '2024-08', '2024-09', '2025-12']


### 6.1 Έλεγχος πληρότητας δεδομένων για τους μήνες 2024-08 / 2024-09

In [10]:
check_cols = ["year_month_str","total_consumption","total_days","counted_supplies","postcode_count"]
display_df = monthly[check_cols].copy()
median_days = display_df["total_days"].median()
median_supplies = display_df["counted_supplies"].median()
for m in ["2024-08","2024-09"]:
    row = display_df[display_df.year_month_str==m].iloc[0]
    print(m, "-> total_days: %.0f%% του τυπικού | counted_supplies: %.0f%% του τυπικού" % (
        100*row.total_days/median_days, 100*row.counted_supplies/median_supplies))
print()
print("=> Η χαμηλή κατανάλωση αυτών των μηνών συνοδεύεται από κατάρρευση κάλυψης καταγραφής,")
print("   όχι μόνο από πτώση ζήτησης — να σημειωθεί ρητά ως πιθανό data-completeness artifact.")


2024-08 -> total_days: 46% του τυπικού | counted_supplies: 56% του τυπικού
2024-09 -> total_days: 12% του τυπικού | counted_supplies: 27% του τυπικού

=> Η χαμηλή κατανάλωση αυτών των μηνών συνοδεύεται από κατάρρευση κάλυψης καταγραφής,
   όχι μόνο από πτώση ζήτησης — να σημειωθεί ρητά ως πιθανό data-completeness artifact.


## 7. Ομαδοποίηση μηνών (k-means, Ward, DTW, feature-based) — 4 μέθοδοι, k=3

In [11]:
def choose_k(X, k_values=range(2,6), method="kmeans"):
    recs=[]
    for k in k_values:
        if method=="kmeans":
            lab = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20).fit_predict(X)
        else:
            lab = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(X)
        if len(set(lab))>1:
            recs.append((k, silhouette_score(X,lab)))
    df = pd.DataFrame(recs, columns=["k","silhouette"])
    return int(df.sort_values("silhouette",ascending=False).iloc[0]["k"]), df

best_k, sil_table = choose_k(Xs, method="kmeans")
print("Βέλτιστο k (μήνες):", best_k); print(sil_table)

monthly["cluster_month_kmeans"] = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=20).fit_predict(Xs)
monthly["cluster_month_hierarchical"] = AgglomerativeClustering(n_clusters=best_k, linkage="ward").fit_predict(Xs)

# DTW: εφαρμόζεται στο 10-διάστατο τυποποιημένο προφίλ κάθε μήνα ως "σχήμα" (Sakoe-Chiba: χωρίς
# περιορισμό warping window· tslearn default). ΣΗΜΕΙΩΣΗ ΜΕΘΟΔΟΥ: εδώ ο άξονας "χρόνου" της DTW
# αντιστοιχεί σε δείκτες συστήματος, όχι σε χρονικές στιγμές — μετρά ομοιότητα σχήματος προφίλ.
X_dtw = Xs.reshape(Xs.shape[0], Xs.shape[1], 1)
dtw_model = TimeSeriesKMeans(n_clusters=best_k, metric="dtw", random_state=RANDOM_STATE, max_iter=20)
monthly["cluster_month_dtw"] = dtw_model.fit_predict(X_dtw)

# Feature-based (μήνες): μεταβλητές ήδη σε επίπεδο χαρακτηριστικού· εφαρμόζεται k-means σε
# PCA(5)-μειωμένο χώρο για ανεξάρτητη 4η οπτική.
pca5 = PCA(n_components=min(5, Xs.shape[1]), random_state=RANDOM_STATE)
X_fb = pca5.fit_transform(Xs)
monthly["cluster_month_feature"] = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=20).fit_predict(X_fb)

for col in ["cluster_month_kmeans","cluster_month_hierarchical","cluster_month_dtw","cluster_month_feature"]:
    sil = silhouette_score(Xs, monthly[col])
    print(f"{col}: silhouette={sil:.3f}")


Βέλτιστο k (μήνες): 2
   k  silhouette
0  2    0.360078
1  3    0.266217
2  4    0.275473
3  5    0.279445


cluster_month_kmeans: silhouette=0.360
cluster_month_hierarchical: silhouette=0.360
cluster_month_dtw: silhouette=0.337
cluster_month_feature: silhouette=0.360


### 7.1 Δείκτες συμφωνίας μεταξύ μεθόδων (ARI / NMI) — μήνες

In [12]:
mm = ["cluster_month_kmeans","cluster_month_hierarchical","cluster_month_dtw","cluster_month_feature"]
print(f"{'':30s} " + " ".join(f"{m[14:]:>14s}" for m in mm))
for a in mm:
    row=[]
    for b in mm:
        row.append(f"{adjusted_rand_score(monthly[a], monthly[b]):.2f}")
    print(f"{a[14:]:30s} " + " ".join(f"{v:>14s}" for v in row))


                                       kmeans   hierarchical            dtw        feature
kmeans                                   1.00           1.00           0.87           1.00
hierarchical                             1.00           1.00           0.87           1.00
dtw                                      0.87           0.87           1.00           0.87
feature                                  1.00           1.00           0.87           1.00


## 8. Ομαδοποίηση ταχυδρομικών κωδίκων (πραγματικές χρονοσειρές κατανάλωσης)

In [13]:
postcode_month = cons_postcode_month.copy()
postcode_matrix = (postcode_month.pivot_table(index=["ΤΚ","Περιοχή"], columns="year_month",
                    values="total_consumption", aggfunc="sum").sort_index(axis=1))
min_nonmissing = max(6, int(0.6*postcode_matrix.shape[1]))
postcode_matrix = postcode_matrix[postcode_matrix.notna().sum(axis=1) >= min_nonmissing].copy()
postcode_matrix_filled = postcode_matrix.T.interpolate(limit_direction="both").T
postcode_matrix_filled = postcode_matrix_filled.apply(lambda r: r.fillna(r.median()), axis=1)
postcode_matrix_filled = postcode_matrix_filled.fillna(postcode_matrix_filled.median())

row_means = postcode_matrix_filled.mean(axis=1)
row_stds  = postcode_matrix_filled.std(axis=1).replace(0,1)
postcode_matrix_z = postcode_matrix_filled.sub(row_means, axis=0).div(row_stds, axis=0)

X_post = postcode_matrix_z.values
best_k_post, sil_post = choose_k(X_post, range(2,7), "kmeans")
print("Βέλτιστο k (ΤΚ):", best_k_post)

postcode_clusters = postcode_matrix_z.copy()
postcode_clusters["cluster_post_kmeans"] = KMeans(n_clusters=best_k_post, random_state=RANDOM_STATE, n_init=20).fit_predict(X_post)
postcode_clusters["cluster_post_hierarchical"] = AgglomerativeClustering(n_clusters=best_k_post, linkage="ward").fit_predict(X_post)

X_post_dtw = X_post[:,:,np.newaxis]
X_post_dtw = TimeSeriesScalerMeanVariance().fit_transform(X_post_dtw)  # confirmed extra step in original code
postcode_clusters["cluster_post_dtw"] = TimeSeriesKMeans(n_clusters=best_k_post, metric="dtw",
                                            random_state=RANDOM_STATE, max_iter=20).fit_predict(X_post_dtw)

def extract_features(mat):
    t = np.arange(mat.shape[1]); feats=[]
    for row in mat:
        slope = np.polyfit(t,row,1)[0]
        diff_std = np.std(np.diff(row))
        acf1 = np.corrcoef(row[:-1], row[1:])[0,1] if len(row)>1 else np.nan
        seasonal_strength = np.var(row - pd.Series(row).rolling(3,min_periods=1).mean())/(np.var(row)+1e-8)
        feats.append([slope, np.mean(row), np.std(row), np.min(row), np.max(row),
                      np.quantile(row,.25), np.quantile(row,.5), np.quantile(row,.75), diff_std, acf1, seasonal_strength])
    return pd.DataFrame(feats, columns=["slope","mean","std","min","max","q25","q50","q75","diff_std","acf1","seasonal_strength"]).fillna(0)

post_feat = extract_features(X_post)
X_post_feat = StandardScaler().fit_transform(post_feat)
postcode_clusters["cluster_post_feature"] = KMeans(n_clusters=best_k_post, random_state=RANDOM_STATE, n_init=20).fit_predict(X_post_feat)
postcode_clusters = postcode_clusters.reset_index()

print(postcode_clusters["cluster_post_kmeans"].value_counts())


Βέλτιστο k (ΤΚ): 2


cluster_post_kmeans
0    707
1    616
Name: count, dtype: int64


### 8.1 Δείκτες συμφωνίας ARI/NMI — ταχυδρομικοί κώδικες

In [14]:
pm = ["cluster_post_kmeans","cluster_post_hierarchical","cluster_post_dtw","cluster_post_feature"]
for i in range(len(pm)):
    for j in range(i+1, len(pm)):
        a,b = pm[i], pm[j]
        ari = adjusted_rand_score(postcode_clusters[a], postcode_clusters[b])
        nmi = normalized_mutual_info_score(postcode_clusters[a], postcode_clusters[b])
        print(f"{a} vs {b}: ARI={ari:.2f} NMI={nmi:.2f}")


cluster_post_kmeans vs cluster_post_hierarchical: ARI=0.52 NMI=0.44
cluster_post_kmeans vs cluster_post_dtw: ARI=0.12 NMI=0.10
cluster_post_kmeans vs cluster_post_feature: ARI=0.11 NMI=0.08
cluster_post_hierarchical vs cluster_post_dtw: ARI=0.23 NMI=0.26
cluster_post_hierarchical vs cluster_post_feature: ARI=0.24 NMI=0.24
cluster_post_dtw vs cluster_post_feature: ARI=0.72 NMI=0.61


## 9. Έξοδος — αρχεία για το άρθρο
Αποθηκεύονται όλοι οι πίνακες που τροφοδοτούν τα Table/Figure του άρθρου.

In [15]:
monthly.to_csv("../results/monthly_results.csv", index=False)
postcode_clusters.to_csv("../results/postcode_results.csv", index=False)
corr_df.to_csv("../results/spearman_correlations.csv", index=False)
print("Έτοιμα αρχεία εξόδου.")


Έτοιμα αρχεία εξόδου.
